# Read file and column

In [ ]:
from pathlib import Path
import pandas as pd
import time

# ============================================================
# READ ONLY 2011 JOB FILES
# NO COMBINE
# NO SAVE
# ============================================================

BASE_DIR = (
    Path.home()
    / "Downloads"
    / "Internship_SCIPE CI-SIP"
    / "MainProject"
    / "2_Job"
    / "JobRawData"
)

FILES = [
    BASE_DIR / "2011-5_May 2011" / "oes_data_2011.xlsx",
    BASE_DIR / "2011-11_November 2011" / "ag_data_2011.xlsx",
]

PREVIEW_ROWS = 10

start_time = time.time()

# ============================================================
# CHECK FILES
# ============================================================

print("FILES TO READ:")
print()

for i, file_path in enumerate(FILES, start=1):
    print(f"{i}. {file_path}")
    print("Exists:", file_path.exists())
    print()

# ============================================================
# READ EACH FILE
# ============================================================

for file_num, file_path in enumerate(FILES, start=1):
    print("=" * 120)
    print(f"FILE {file_num}: {file_path.name}")
    print("FULL PATH:")
    print(file_path)
    print("=" * 120)

    if not file_path.exists():
        print("ERROR: File does not exist.")
        continue

    try:
        # Open Excel file without loading all sheets fully
        xls = pd.ExcelFile(file_path, engine="openpyxl")

        print("Sheet names:")
        for sheet in xls.sheet_names:
            print("-", sheet)

        print()
        print("Total sheets:", len(xls.sheet_names))
        print()

        # Read each sheet preview only
        for sheet_name in xls.sheet_names:
            print("-" * 120)
            print("SHEET:", sheet_name)

            # Read header only for full column names
            header_df = pd.read_excel(
                file_path,
                sheet_name=sheet_name,
                nrows=0,
                dtype=str,
                engine="openpyxl"
            )

            print("Columns count:", len(header_df.columns))
            print("Columns:")
            for col in header_df.columns:
                print(col)

            print()

            # Read preview rows only
            preview_df = pd.read_excel(
                file_path,
                sheet_name=sheet_name,
                nrows=PREVIEW_ROWS,
                dtype=str,
                engine="openpyxl"
            )

            print(f"Preview first {PREVIEW_ROWS} rows:")
            display(preview_df)

    except Exception as e:
        print("ERROR reading file:")
        print(e)

# ============================================================
# DONE
# ============================================================

elapsed = time.time() - start_time

print()
print("=" * 120)
print("DONE READING ONLY")
print("No file was saved.")
print(f"Total time: {elapsed:.2f} seconds")

# Join 2011

In [ ]:
from pathlib import Path
from openpyxl import load_workbook, Workbook
import time

# ============================================================
# COMBINE 2011 JOB DATA INTO ONE EXCEL FILE
# MEMORY OPTIMIZED VERSION
#
# Output:
# /Users/YourUserName123/Downloads/all_data_2011.xlsx
# ============================================================

start_time = time.time()

# ============================================================
# PATH SETTINGS
# ============================================================

BASE_DIR = (
    Path.home()
    / "Downloads"
    / "Internship_SCIPE CI-SIP"
    / "MainProject"
    / "2_Job"
    / "JobRawData"
)

OUTPUT_FILE = Path.home() / "Downloads" / "all_data_2011.xlsx"

FILES_TO_COMBINE = [
    {
        "file_path": BASE_DIR / "2011-5_May 2011" / "oes_data_2011.xlsx",
        "sheet_name": "oes_data_2011",
        "source_period": "May 2011",
    },
    {
        "file_path": BASE_DIR / "2011-11_November 2011" / "ag_data_2011.xlsx",
        "sheet_name": "all data ag file, nov 2011",
        "source_period": "November 2011 Agriculture",
    },
]

# ============================================================
# FINAL COLUMN ORDER
# Union of May 2011 + November 2011 agriculture columns
# All lowercase for consistency
# ============================================================

FINAL_COLUMNS = [
    "source_file",
    "source_folder",
    "source_sheet",
    "source_period",

    "area",
    "area_title",
    "area_type",
    "naics",
    "naics_title",
    "own_code",
    "occ_code",
    "occ_title",
    "group",
    "occ_group",
    "tot_emp",
    "emp_prse",
    "jobs_1000",
    "loc_q",
    "pct_tot",
    "h_mean",
    "a_mean",
    "mean_prse",
    "h_pct10",
    "h_pct25",
    "h_median",
    "h_pct75",
    "h_pct90",
    "a_pct10",
    "a_pct25",
    "a_median",
    "a_pct75",
    "a_pct90",
    "annual",
    "hourly",
]

# ============================================================
# CREATE OUTPUT WORKBOOK
# write_only=True keeps memory low
# ============================================================

out_wb = Workbook(write_only=True)
out_ws = out_wb.create_sheet("all_data_2011")

# Write header
out_ws.append(FINAL_COLUMNS)

total_rows_written = 0

# ============================================================
# READ EACH FILE ONE AT A TIME
# read_only=True keeps memory low
# ============================================================

for info in FILES_TO_COMBINE:
    file_path = info["file_path"]
    sheet_name = info["sheet_name"]
    source_period = info["source_period"]

    print("=" * 80)
    print(f"Reading file: {file_path}")
    print(f"Sheet: {sheet_name}")

    if not file_path.exists():
        print(f"ERROR: File not found: {file_path}")
        continue

    wb = load_workbook(
        filename=file_path,
        read_only=True,
        data_only=True
    )

    if sheet_name not in wb.sheetnames:
        print(f"ERROR: Sheet not found: {sheet_name}")
        print(f"Available sheets: {wb.sheetnames}")
        wb.close()
        continue

    ws = wb[sheet_name]

    # Read header row
    rows = ws.iter_rows(values_only=True)
    header_row = next(rows)

    # Normalize headers to lowercase
    source_headers = []
    for col in header_row:
        if col is None:
            source_headers.append("")
        else:
            source_headers.append(str(col).strip().lower())

    # Map header name to column index
    header_index = {
        col_name: idx
        for idx, col_name in enumerate(source_headers)
        if col_name != ""
    }

    rows_written_for_file = 0

    # Process data rows
    for row in rows:
        output_row = []

        for final_col in FINAL_COLUMNS:
            if final_col == "source_file":
                output_row.append(file_path.name)

            elif final_col == "source_folder":
                output_row.append(file_path.parent.name)

            elif final_col == "source_sheet":
                output_row.append(sheet_name)

            elif final_col == "source_period":
                output_row.append(source_period)

            else:
                # If the source file has this column, use it.
                # If not, leave blank.
                if final_col in header_index:
                    idx = header_index[final_col]
                    value = row[idx] if idx < len(row) else None
                    output_row.append(value)
                else:
                    output_row.append(None)

        out_ws.append(output_row)

        rows_written_for_file += 1
        total_rows_written += 1

        if rows_written_for_file % 50000 == 0:
            print(f"  Written {rows_written_for_file:,} rows from this file...")

    wb.close()

    print(f"Finished: {file_path.name}")
    print(f"Rows written from this file: {rows_written_for_file:,}")

# ============================================================
# SAVE OUTPUT FILE
# ============================================================

print("=" * 80)
print("Saving combined Excel file...")
out_wb.save(OUTPUT_FILE)

end_time = time.time()

print("=" * 80)
print("DONE")
print(f"Saved file:")
print(OUTPUT_FILE)
print(f"Total rows written: {total_rows_written:,}")
print(f"Total time: {end_time - start_time:.2f} seconds")